# AnnDataset: PyTorch Integration for anndata objects

This notebook demonstrates how to use `AnnDataset` for machine learning with anndata objects in PyTorch.

## Setup


In [1]:
import numpy as np
import scanpy as sc
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch import nn, optim
from torch.utils.data import DataLoader

from anndata.experimental.pytorch import AnnDataset, Compose, Transform

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cpu


## Load Data


In [2]:
# Load example data
adata = sc.datasets.pbmc3k_processed()

print(f"Data shape: {adata.shape}")
print(f"Cell types: {adata.obs['louvain'].unique()}")

# For large datasets stored on disk, use:
# dataset = AnnDataset("/path/to/dataset.h5ad", transform=transforms)


Data shape: (2638, 1838)
Cell types: ['CD4 T cells', 'B cells', 'CD14+ Monocytes', 'NK cells', 'CD8 T cells', 'FCGR3A+ Monocytes', 'Dendritic cells', 'Megakaryocytes']
Categories (8, object): ['CD4 T cells', 'CD14+ Monocytes', 'B cells', 'CD8 T cells', 'NK cells', 'FCGR3A+ Monocytes', 'Dendritic cells', 'Megakaryocytes']


## Create Custom Transforms

In [ ]:
class NormalizeRowSum(Transform):
    """Normalize counts to target sum per cell."""
    def __init__(self, target_sum=1e4):
        self.target_sum = target_sum

    def __call__(self, data_dict):
        X = data_dict["X"]
        X = torch.clamp(X, min=0)
        row_sum = torch.sum(X, dim=-1, keepdim=True) + 1e-8
        data_dict["X"] = X * (self.target_sum / row_sum)
        return data_dict

    def __repr__(self):
        return f"NormalizeRowSum(target_sum={self.target_sum})"

class LogTransform(Transform):
    """Apply log1p transformation."""
    def __call__(self, data_dict):
        data_dict["X"] = torch.log1p(data_dict["X"])
        return data_dict

class EncodeLabels(Transform):
    """Encode string labels to integers."""
    def __init__(self, obs_key='louvain', encoded_key=None, label_encoder=None):
        self.obs_key = obs_key
        self.encoded_key = encoded_key or f"obs_{obs_key}_encoded"
        self.label_encoder = label_encoder

    def __call__(self, data_dict):
        # Access metadata using obs_{column_name} key pattern
        obs_key = f"obs_{self.obs_key}"

        if obs_key in data_dict and self.label_encoder is not None:
            # Get string label from metadata and encode it
            string_label = data_dict[obs_key]
            encoded_label = self.label_encoder.transform([string_label])[0]
            data_dict[self.encoded_key] = torch.tensor(encoded_label, dtype=torch.long)
        return data_dict

    def __repr__(self):
        return f"EncodeLabels(obs_key='{self.obs_key}', encoded_key='{self.encoded_key}')"


## Create Train/Test Datasets


In [4]:
# First do train/test split (before fitting label encoder)
all_labels_str = adata.obs['louvain'].values
train_idx, test_idx = train_test_split(
    np.arange(len(adata)), test_size=0.2, random_state=42, stratify=all_labels_str
)

# Fit label encoder ONLY on training data
train_labels_str = all_labels_str[train_idx]
test_labels_str = all_labels_str[test_idx]
label_encoder = LabelEncoder()
label_encoder.fit(train_labels_str)
n_classes = len(label_encoder.classes_)
print(f"Label encoder classes: {label_encoder.classes_}")

# Create complete transform pipeline with label encoding
complete_transforms = Compose([
    NormalizeRowSum(target_sum=1e4),
    LogTransform(),
    EncodeLabels(obs_key='louvain', label_encoder=label_encoder)
])

train_dataset = AnnDataset(adata, obs_subset=train_idx, transform=complete_transforms)
test_dataset = AnnDataset(adata, obs_subset=test_idx, transform=complete_transforms)

print(f"\nTraining samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Features: {train_dataset.shape[1]}")
print(f"Classes: {n_classes}")
print(f"Complete transform pipeline: {complete_transforms}")


Label encoder classes: ['B cells' 'CD14+ Monocytes' 'CD4 T cells' 'CD8 T cells' 'Dendritic cells'
 'FCGR3A+ Monocytes' 'Megakaryocytes' 'NK cells']

Training samples: 2110
Test samples: 528
Features: 1838
Classes: 8
Complete transform pipeline: Compose(
    NormalizeRowSum(target_sum=10000.0)
    LogTransform()
    EncodeLabels(obs_key='louvain', encoded_key='obs_louvain_encoded')
)


## Create DataLoaders


In [5]:
# Standard DataLoaders (for in-memory data)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

# For disk-based data, use optimized DataLoaders:
# train_loader = train_dataset.get_optimized_dataloader(batch_size=64, shuffle=True)
# OR
# train_loader = DataLoader(train_dataset, batch_size=64, collate_fn=train_dataset.get_collate_fn())

print(f"Training batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

# Test data loading
sample_batch = next(iter(train_loader))
print(f"Batch keys: {list(sample_batch.keys())}")
print(f"X shape: {sample_batch['X'].shape}")
print(f"Available metadata: {[k for k in sample_batch.keys() if k.startswith('obs_')]}")


Training batches: 33
Test batches: 9
Batch keys: ['X', 'obs_n_genes', 'obs_percent_mito', 'obs_n_counts', 'obs_louvain', 'obs_louvain_encoded']
X shape: torch.Size([64, 1838])
Available metadata: ['obs_n_genes', 'obs_percent_mito', 'obs_n_counts', 'obs_louvain', 'obs_louvain_encoded']


##  Batch Loading

In [6]:
for batch_idx, batch in enumerate(train_loader):
    X = batch["X"]
    labels_encoded = batch["obs_louvain_encoded"]
    print(f"Batch {batch_idx}: X shape = {X.shape}, labels shape = {labels_encoded.shape}")
    if batch_idx == 2:
        break  # Only show first 3 batches


Batch 0: X shape = torch.Size([64, 1838]), labels shape = torch.Size([64])
Batch 1: X shape = torch.Size([64, 1838]), labels shape = torch.Size([64])
Batch 2: X shape = torch.Size([64, 1838]), labels shape = torch.Size([64])
